In [ ]:

import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

In [ ]:
PROJECT_NAME = "ASL_Signer_Independent_Multimodal_Model"

DATASET_PATH = "/content/asl_dataset"
OUTPUT_PATH = "/content/asl_outputs"

RANDOM_SEED = 42

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Project:", PROJECT_NAME)
print("Dataset folder:", DATASET_PATH)
print("Output folder:", OUTPUT_PATH)
print("Random seed:", RANDOM_SEED)

print("Project configuration completed!")

In [ ]:
print("Dataset folder exists:", os.path.exists(DATASET_PATH))
print("Output folder exists:", os.path.exists(OUTPUT_PATH))

print("Dataset folder contents:", os.listdir(DATASET_PATH))
print("Output folder contents:", os.listdir(OUTPUT_PATH))

In [ ]:
import json

sample_record = {
    "video_id": "example_001",
    "gloss": "hello",
    "signer_id": 1
}

print(json.dumps(sample_record, indent=4))

In [ ]:
import urllib.request

ANNOTATION_URL = (
    "https://raw.githubusercontent.com/dxli94/WLASL/master/"
    "start_kit/WLASL_v0.3.json"
)

ANNOTATION_PATH = os.path.join(
    DATASET_PATH,
    "WLASL_v0.3.json"
)

try:
    urllib.request.urlretrieve(
        ANNOTATION_URL,
        ANNOTATION_PATH
    )

    print("Annotation file downloaded successfully!")
    print("Saved at:", ANNOTATION_PATH)

except Exception as e:
    print("Download failed:", e)

In [ ]:
with open(ANNOTATION_PATH, "r", encoding="utf-8") as file:
    annotations = json.load(file)

print("Annotation file loaded successfully!")
print("Number of sign categories:", len(annotations))
print("First sign category:", annotations[0]["gloss"])
print("Keys in first category:", annotations[0].keys())

In [ ]:
print("First annotation record:")
print(json.dumps(annotations[0], indent=4))

In [ ]:
first_category = annotations[0]

print("Sign label:", first_category["gloss"])
print("Number of video instances:", len(first_category["instances"]))

if first_category["instances"]:
    print("First video instance:")
    print(json.dumps(first_category["instances"][0], indent=4))

In [ ]:
total_categories = len(annotations)

all_instances = []

for category in annotations:
    for instance in category["instances"]:
        all_instances.append({
            "gloss": category["gloss"],
            "video_id": instance.get("video_id"),
            "signer_id": instance.get("signer_id"),
            "split": instance.get("split")
        })

df = pd.DataFrame(all_instances)

print("Total sign categories:", total_categories)
print("Total annotated video instances:", len(df))
print("Unique signers:", df["signer_id"].nunique())

print("\nVideos per split:")
print(df["split"].value_counts(dropna=False))

print("\nFirst five video records:")
print(df.head())

In [ ]:
split_signers = {}

for split_name in df["split"].dropna().unique():
    split_data = df[df["split"] == split_name]

    split_signers[split_name] = set(
        split_data["signer_id"].dropna()
    )

print("Unique signers in each split:")

for split_name, signers in split_signers.items():
    print(split_name, ":", len(signers), "signers")

print("\nSigner overlap between splits:")

split_names = list(split_signers.keys())

for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        split_a = split_names[i]
        split_b = split_names[j]

        overlap = split_signers[split_a] & split_signers[split_b]

        print(f"{split_a} vs {split_b}: {len(overlap)} shared signers")

In [ ]:
top_categories = (
    df["gloss"]
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10, 5))

top_categories.plot(kind="bar")

plt.title("Top 10 ASL Sign Categories by Annotated Videos")
plt.xlabel("Sign Category")
plt.ylabel("Number of Video Instances")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
print("Missing values in each column:\n")
print(df.isnull().sum())

print("\nDuplicate video IDs:", df["video_id"].duplicated().sum())

print("\nTotal annotation records:", len(df))

print("\nRecords with missing signer IDs:",
      df["signer_id"].isnull().sum())

print("\nRecords with missing video IDs:",
      df["video_id"].isnull().sum())

print("\nRecords with missing sign labels:",
      df["gloss"].isnull().sum())

In [ ]:
category_stats = (
    df.groupby("gloss")
    .agg(
        video_count=("video_id", "count"),
        signer_count=("signer_id", "nunique")
    )
    .sort_values("video_count", ascending=False)
)

print("Top 20 sign categories:")
print(category_stats.head(20))

In [ ]:
candidate_categories = category_stats[
    (category_stats["video_count"] >= 20) &
    (category_stats["signer_count"] >= 3)
]

print("Candidate categories:", len(candidate_categories))
print(candidate_categories.head(20))

In [ ]:
candidate_labels = candidate_categories.index

candidate_df = df[df["gloss"].isin(candidate_labels)].copy()

split_counts = pd.crosstab(
    candidate_df["gloss"],
    candidate_df["split"]
)

print("Number of annotated videos per sign and split:")
print(split_counts.head(20))

In [ ]:
required_splits = {"train", "val", "test"}

available_splits = {
    str(split).lower()
    for split in candidate_df["split"].dropna().unique()
}

print("Available split names:", available_splits)

if required_splits.issubset(available_splits):
    category_split_presence = split_counts.reindex(
        columns=["train", "val", "test"],
        fill_value=0
    )

    complete_categories = category_split_presence[
        (category_split_presence > 0).all(axis=1)
    ]

    print("\nCategories present in all three splits:",
          len(complete_categories))

    print(complete_categories.head(20))
else:
    print(
        "\nThe split names differ from the expected names. "
        "Inspect the output before proceeding."
    )

In [ ]:
split_signer_sets = {}

for split_name in candidate_df["split"].dropna().unique():
    split_data = candidate_df[
        candidate_df["split"] == split_name
    ]

    split_signer_sets[split_name] = set(
        split_data["signer_id"].dropna()
    )

print("Signer counts per split:")

for split_name, signer_ids in split_signer_sets.items():
    print(f"{split_name}: {len(signer_ids)} unique signers")

print("\nPairwise signer overlap:")

split_names = list(split_signer_sets.keys())

for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        split_a = split_names[i]
        split_b = split_names[j]

        shared_signers = (
            split_signer_sets[split_a]
            & split_signer_sets[split_b]
        )

        print(
            f"{split_a} vs {split_b}: "
            f"{len(shared_signers)} shared signers"
        )

In [ ]:
summary = candidate_df.groupby("gloss").agg(
    total_videos=("video_id", "count"),
    unique_signers=("signer_id", "nunique")
)

for split_name in ["train", "val", "test"]:
    split_data = candidate_df[
        candidate_df["split"].str.lower() == split_name
    ]

    split_video_counts = split_data.groupby("gloss")["video_id"].count()

    summary[f"{split_name}_videos"] = (
        split_video_counts.reindex(summary.index, fill_value=0)
    )

    split_signer_counts = split_data.groupby("gloss")["signer_id"].nunique()

    summary[f"{split_name}_signers"] = (
        split_signer_counts.reindex(summary.index, fill_value=0)
    )

summary = summary.sort_values(
    by="total_videos",
    ascending=False
)

print("Research dataset summary:")
display(summary.head(20))

In [ ]:
required_splits = ["train", "val", "test"]

signer_sets = {}

for split_name in required_splits:
    split_data = candidate_df[
        candidate_df["split"].astype(str).str.lower().str.strip()
        == split_name
    ]

    signer_sets[split_name] = set(
        split_data["signer_id"].dropna()
    )

print("SIGNER INDEPENDENCE CHECK")
print("-" * 35)

for split_name, signers in signer_sets.items():
    print(f"{split_name}: {len(signers)} unique signers")

print("\nOverlap between splits:")

for i in range(len(required_splits)):
    for j in range(i + 1, len(required_splits)):
        split_a = required_splits[i]
        split_b = required_splits[j]

        shared = signer_sets[split_a] & signer_sets[split_b]

        print(
            f"{split_a} vs {split_b}: "
            f"{len(shared)} shared signers"
        )

if all(
    len(signer_sets[a] & signer_sets[b]) == 0
    for i, a in enumerate(required_splits)
    for b in required_splits[i + 1:]
):
    print("\nResult: No signer overlap detected.")
else:
    print

In [ ]:
import os

print("Checking dataset files...")
print("-" * 35)

print("Dataset folder:", DATASET_PATH)
print("\nFiles in dataset folder:")

for item in os.listdir(DATASET_PATH):
    item_path = os.path.join(DATASET_PATH, item)

    if os.path.isfile(item_path):
        size_mb = os.path.getsize(item_path) / (1024 * 1024)
        print(f"FILE: {item} ({size_mb:.2f} MB)")
    else:
        print(f"FOLDER: {item}")

video_extensions = (".mp4", ".avi", ".mov", ".webm")

video_files = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(video_extensions):
            video_files.append(os.path.join(root, file))

print("\nTotal video files found:", len(video_files))

if len(video_files) == 0:
    print("Actual video files have not been found in the dataset folder.")
else:
    print("Example video paths:")
    for path in video_files[:5]:
        print(path)

In [ ]:
import json

with open(ANNOTATION_PATH, "r", encoding="utf-8") as file:
    annotations = json.load(file)

print("First 3 video instances:\n")

count = 0

for category in annotations:
    for instance in category["instances"]:
        print(json.dumps(instance, indent=4))

        count += 1

        if count == 3:
            break

    if count == 3:
        break

In [ ]:
from urllib.parse import urlparse, unquote
import re

video_records = []

for category in annotations:
    for instance in category["instances"]:
        raw_url = instance.get("url", "")

        # Extract the actual URL if it is wrapped in Markdown
        match = re.search(r"https?://[^\s\]]+", raw_url)

        video_url = match.group(0) if match else raw_url

        video_records.append({
            "gloss": category["gloss"],
            "video_id": instance.get("video_id"),
            "signer_id": instance.get("signer_id"),
            "split": instance.get("split"),
            "source": instance.get("source"),
            "url": video_url
        })

video_df = pd.DataFrame(video_records)

print("Total video records:", len(video_df))
print("\nVideo sources:")
print(video_df["source"].value_counts(dropna=False))

print("\nFirst 5 extracted URLs:")
display(video_df[["gloss", "video_id", "source", "url"]].head())

In [ ]:
import requests

test_url = video_df.loc[
    video_df["url"].astype(str).str.startswith("http"),
    "url"
].iloc[0]

print("Testing URL:")
print(test_url)

try:
    response = requests.get(
        test_url,
        stream=True,
        timeout=15,
        allow_redirects=True
    )

    print("\nHTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    print("Final URL:", response.url)

    if response.status_code == 200:
        print("\nThe server responded successfully.")
    else:
        print("\nThe server did not return HTTP 200.")

    response.close()

except requests.RequestException as error:
    print("\nRequest failed:", error)

In [ ]:
import os
import requests
from urllib.parse import urlparse

VIDEO_PATH = os.path.join(DATASET_PATH, "videos")
os.makedirs(VIDEO_PATH, exist_ok=True)

test_record = video_df[
    video_df["url"].astype(str).str.contains("aslbricks.org", na=False)
].iloc[0]

test_url = test_record["url"].strip()

if test_url.startswith("["):
    test_url = test_url.split("](")[-1].rstrip(")")

video_id = test_record["video_id"]
video_file = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

response = requests.get(test_url, timeout=30)
response.raise_for_status()

if "video/" not in response.headers.get("Content-Type", "").lower():
    raise ValueError("The downloaded response does not appear to be a video.")

with open(video_file, "wb") as file:
    file.write(response.content)

print("Video downloaded successfully!")
print("Video ID:", video_id)
print("Saved to:", video_file)
print("File size (MB):", round(os.path.getsize(video_file) / (1024 * 1024), 2))